<a href="https://colab.research.google.com/github/martinthuriaux/Auto-Interp-Causal-Validation/blob/main/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook will check that the environment works and then mounts the drive, donwloads the image datasets (ImageNetV2, DTD, Pets and Flowers102), alongside loading ResNet-18, whilst confirming that all 1920 units are available to study.

This is designed so that no costs should be insured.

<a href="https://colab.research.google.com/github/martinthuriaux/CNN-Watch-vs-Do/blob/main/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Setup and cold-start smoke test

A fresh Colab session can mount Drive,
download all four datasets to the session's *local* disk, and load ResNet-18 all in
under 15 minutes.

This notebook is idempotent: re-running any cell is safe, and later notebooks re-use the
`quick_setup()` pattern defined here at the top of every session.

Datasets (all free, no registration):

| Dataset | Role | Expected images |
|---|---|---|
| ImageNetV2 (matched-frequency) | exemplar mining (half) + watching-test holdout (half) + doing-test evaluation | 10,000 |
| DTD (textures) | exemplar mining | 5,640 |
| Oxford-IIIT Pets | exemplar mining | 7,349 |
| Flowers102 | exemplar mining | 8,189 |

All datasets live on the session's local disk at `/content/data`
and are re-fetched each session. Drive holds **results only** except a cached copy of the ImageNetV2 tarball, because copying inside Google's
network is much faster than re-downloading it.

In [ ]:
import os, time, json, hashlib, shutil, tarfile, urllib.request
from pathlib import Path

T0 = time.time()
timings = {}
def tick(name, t_start):
    timings[name] = time.time() - t_start
    print(f"[{time.time()-T0:6.1f}s total] {name}: {timings[name]:.1f}s")


## 1. Mount Drive and create the results layout

In [ ]:
t = time.time()
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/cnn_watch_vs_do")
RESULTS = PROJECT_ROOT / "results"
LAYOUT = [
    RESULTS / "activations",   # per-stage activation shards (parquet)
    RESULTS / "exemplars",     # top-16 tracker checkpoints + exemplar grids
    RESULTS / "labels",        # labeler A/B outputs
    RESULTS / "scores",        # watching/doing score tables
    RESULTS / "figures",
    PROJECT_ROOT / "cache",    # sanctioned dataset cache (ImageNetV2 tarball only)
]
for p in LAYOUT:
    p.mkdir(parents=True, exist_ok=True)
print("Drive layout ready under", PROJECT_ROOT)
tick("mount_drive_and_layout", t)


Mounted at /content/drive
Drive layout ready under /content/drive/MyDrive/cnn_watch_vs_do
[  66.6s total] mount_drive_and_layout: 66.6s


## 2. Frozen configuration

Written to Drive once; later notebooks load it and refuse to run if their local constants disagree

In [ ]:
CONFIG = {
    "project": "cnn-watch-vs-do",
    "plan_version": "v1.0 (committed PDF)",
    "seed": 0,

    # Model and units (PLAN.md §3)
    "model": "resnet18",
    "weights": "IMAGENET1K_V1",
    "stages": ["layer1.0", "layer1.1", "layer2.0", "layer2.1",
               "layer3.0", "layer3.1", "layer4.0", "layer4.1"],
    "expected_stage_channels": [64, 64, 128, 128, 256, 256, 512, 512],
    "n_units": 1920,
    "activation_primary": "spatial_max",   # spatial_mean recorded for robustness

    # Data
    "datasets": ["imagenetv2", "dtd", "pets", "flowers102"],
    "expected_counts": {"imagenetv2": 10000, "dtd": 5640, "pets": 7349, "flowers102": 8189},
    "imagenetv2_split_seed": 0,            # 50/50 mining vs watching-holdout split
    "input_size": 224,

    # Frozen thresholds (PLAN.md §7 — do not touch after week 0)
    "top_firing_quantile": 0.01,           # unit's top-firing set = top 1% of scoring images
    "n_exemplars": 16,
    "exemplar_crop_px": 128,
    "screening_null_percentile": 95,       # doing-test screening pass criterion
    "high_watching_quantile": 0.25,        # 'high watching' = top quartile
    "h3_weak_rho": 0.3,
    "n_permutations": 1000,
    "fdr_alpha": 0.05,
    "checkpoint_every_images": 5000,
}

cfg_path = RESULTS / "config.json"
if cfg_path.exists():
    on_disk = json.load(open(cfg_path))
    assert on_disk == CONFIG, ("config.json on Drive differs from this notebook — "
        "either restore the notebook constants or record a written amendment in PLAN.md")
    print("config.json matches (already frozen)")
else:
    json.dump(CONFIG, open(cfg_path, "w"), indent=2)
    print("config.json written (frozen)")

CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
print("config hash:", CONFIG_HASH)


config.json matches (already frozen)
config hash: 804f511675ac


## 3. Datasets to local disk

`/content/data` is wiped when the session dies, rebuilt by this cell next session.

In [ ]:
t = time.time()
DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

# ---------- 3a. ImageNetV2 (matched-frequency), with Drive tarball cache ----------
INV2_URL = ("https://huggingface.co/datasets/vaishaal/ImageNetV2/"
            "resolve/main/imagenetv2-matched-frequency.tar.gz")
INV2_TAR_CACHE = PROJECT_ROOT / "cache" / "imagenetv2-matched-frequency.tar.gz"
INV2_DIR = DATA / "imagenetv2-matched-frequency-format-val"

if not INV2_DIR.exists():
    local_tar = DATA / "imagenetv2.tar.gz"
    if INV2_TAR_CACHE.exists():
        print("ImageNetV2: copying cached tarball from Drive...")
        shutil.copy(INV2_TAR_CACHE, local_tar)
    else:
        print("ImageNetV2: downloading (~1.2 GB, first time only)...")
        urllib.request.urlretrieve(INV2_URL, local_tar)
        print("ImageNetV2: caching tarball to Drive for future sessions...")
        shutil.copy(local_tar, INV2_TAR_CACHE)
    print("ImageNetV2: extracting...")
    with tarfile.open(local_tar) as tf:
        tf.extractall(DATA)
    local_tar.unlink()
assert INV2_DIR.exists(), f"expected {INV2_DIR} after extraction — inspect the tarball layout"
tick("imagenetv2", t)


ImageNetV2: copying cached tarball from Drive...
ImageNetV2: extracting...


/tmp/ipykernel_985/2121099772.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(DATA)


[ 103.1s total] imagenetv2: 35.9s


In [ ]:
t = time.time()
# ---------- 3b. DTD, Pets, Flowers102 via torchvision ----------
import torchvision
from torchvision import datasets as tvd

# All splits are downloaded so total counts match the plan.
dtd = [tvd.DTD(root=DATA, split=s, download=True) for s in ["train", "val", "test"]]
pets = [tvd.OxfordIIITPet(root=DATA, split=s, download=True) for s in ["trainval", "test"]]
flowers = [tvd.Flowers102(root=DATA, split=s, download=True) for s in ["train", "val", "test"]]
tick("dtd_pets_flowers", t)


100%|██████████| 625M/625M [00:40<00:00, 15.4MB/s]
100%|██████████| 792M/792M [00:54<00:00, 14.5MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 9.31MB/s]
100%|██████████| 345M/345M [00:18<00:00, 18.2MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.68MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 24.5MB/s]


[ 273.5s total] dtd_pets_flowers: 170.3s


## 4. Count verification against the plan

In [ ]:
from glob import glob

counts = {
    "imagenetv2": len(glob(str(INV2_DIR / "*" / "*.jpeg"))),
    "dtd": sum(len(d) for d in dtd),
    "pets": sum(len(d) for d in pets),
    "flowers102": sum(len(d) for d in flowers),
}
print(f"{'dataset':<12} {'found':>7} {'expected':>9}")
ok = True
for k, v in counts.items():
    exp = CONFIG["expected_counts"][k]
    flag = "OK" if v == exp else "MISMATCH"
    ok &= (v == exp)
    print(f"{k:<12} {v:>7} {exp:>9}   {flag}")
assert ok, "dataset counts differ from PLAN.md — investigate before proceeding"


dataset        found  expected
imagenetv2     10000     10000   OK
dtd             5640      5640   OK
pets            7349      7349   OK
flowers102      8189      8189   OK


## 5. Model load and test batch

In [ ]:
t = time.time()
import torch
from torchvision.models import resnet18, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "none")

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()   # the frozen 224px pipeline for ALL notebooks

# Verify the unit inventory matches the plan: 8 residual-block outputs.
stage_channels = []
hooks, seen = [], {}
def mk_hook(name):
    def h(module, inp, out): seen[name] = out.shape
    return h
for name in CONFIG["stages"]:
    layer_name, block_idx = name.split(".")
    block = getattr(model, layer_name)[int(block_idx)]
    hooks.append(block.register_forward_hook(mk_hook(name)))

with torch.no_grad():
    x = torch.randn(4, 3, 224, 224).to(device)
    logits = model(x)
for h in hooks: h.remove()

stage_channels = [seen[s][1] for s in CONFIG["stages"]]
print("stage channels:", stage_channels)
assert stage_channels == CONFIG["expected_stage_channels"]
assert sum(stage_channels) == CONFIG["n_units"] == 1920
assert logits.shape == (4, 1000)
print("unit inventory verified: 1,920 units across 8 stages")
tick("model_and_test_batch", t)


device: cuda | GPU: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


stage channels: [64, 64, 128, 128, 256, 256, 512, 512]
unit inventory verified: 1,920 units across 8 stages
[ 275.5s total] model_and_test_batch: 2.0s


## 6. Cold-start acceptance verdict

In [ ]:
total_min = (time.time() - T0) / 60
print(f"{'step':<28} {'seconds':>8}")
for k, v in timings.items():
    print(f"{k:<28} {v:>8.1f}")
print("-" * 38)
print(f"{'TOTAL':<28} {total_min*60:>8.1f}  ({total_min:.1f} min)")

checks = {
    "Drive mounted + layout": RESULTS.exists(),
    "config.json frozen": (RESULTS / "config.json").exists(),
    "all four datasets present": ok,
    "GPU available": torch.cuda.is_available(),
    "1,920-unit inventory verified": sum(stage_channels) == 1920,
    "under 15 minutes": total_min < 15,
}
print()
for name, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

if all(checks.values()):
    print("\nCOLD-START ACCEPTANCE: PASS — Stage 0 complete.")
else:
    print("\nCOLD-START ACCEPTANCE: FAIL — fix the failing item before starting week 1.")
    if total_min >= 15 and timings.get("imagenetv2", 0) > 300:
        print("Hint: the ImageNetV2 download dominated. The Drive tarball cache is now "
              "populated, so the NEXT cold start will be much faster — rerun in a fresh "
              "session to confirm the criterion honestly.")


step                          seconds
mount_drive_and_layout           66.6
imagenetv2                       35.9
dtd_pets_flowers                170.3
model_and_test_batch              2.0
--------------------------------------
TOTAL                           275.5  (4.6 min)

  [PASS] Drive mounted + layout
  [PASS] config.json frozen
  [PASS] all four datasets present
  [PASS] GPU available
  [PASS] 1,920-unit inventory verified
  [PASS] under 15 minutes

COLD-START ACCEPTANCE: PASS — Stage 0 complete.
